In [9]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from time import sleep

# === CONFIG ===

# Output CSV path
output_csv = r"D:\Android_Mobile_App\AndroidProject_5th\github_android_search_results.csv"

# Load GitHub token
load_dotenv("All_Tokens.env")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise ValueError("❌ GITHUB_TOKEN not found in .env")

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json"
}

# Searching keywords and exclusions
#toy_keywords = ["tutorial", "sample", "test", "example", "hello", "playground", "dummy", "sandbox"]
toy_keywords = ["tutorial", "sample"]
#base = "android path:AndroidManifest.xml stars:>0"
base = "android stars:>0"
#exclusion = ""
exclusion = " ".join(f"-in:name {kw} -in:description {kw} " for kw in toy_keywords)


queries = [
    f"{base} language:Kotlin {exclusion}",
    f"{base} language:Java {exclusion}",
    f"{base} language:Dart {exclusion}"
]

# === Function for paginated search ===

def run_search(query):
    all_items = []
    per_page = 100  # GitHub max
    max_pages = 10  # 10 pages × 100 = 1,000 max results

    for page in range(1, max_pages + 1):
        print(f" Querying page {page} for: {query[:50]}...")
        params = {
            "q": query,
            "per_page": per_page,
            "page": page
        }
        response = requests.get("https://api.github.com/search/repositories",
                                headers=HEADERS, params=params)

        if response.status_code != 200:
            print(f"❌ API error: {response.status_code} — {response.text}")
            break

        data = response.json()
        items = data.get("items", [])
        if not items:
            break

        all_items.extend(items)

        # Be polite: avoid secondary rate limit
        sleep(2)

    print(f"✅ Retrieved {len(all_items)} results for query.")
    return all_items

# === Run all queries ===

all_results = []
for q in queries:
    items = run_search(q)
    all_results.extend(items)

# === Normalize JSON to DataFrame ===

df = pd.json_normalize(all_results)

# === Save to CSV ===

os.makedirs(os.path.dirname(output_csv), exist_ok=True)
df.to_csv(output_csv, index=False)
print(f"✅ All results saved to: {output_csv}")


 Querying page 1 for: android stars:>0 language:Kotlin -in:name tutorial...
 Querying page 2 for: android stars:>0 language:Kotlin -in:name tutorial...
✅ Retrieved 41 results for query.
 Querying page 1 for: android stars:>0 language:Java -in:name tutorial -...
 Querying page 2 for: android stars:>0 language:Java -in:name tutorial -...
 Querying page 3 for: android stars:>0 language:Java -in:name tutorial -...
✅ Retrieved 121 results for query.
 Querying page 1 for: android stars:>0 language:Dart -in:name tutorial -...
 Querying page 2 for: android stars:>0 language:Dart -in:name tutorial -...
✅ Retrieved 2 results for query.
✅ All results saved to: D:\Android_Mobile_App\AndroidProject_5th\github_android_search_results.csv


In [ ]:
import requests
import pandas as pd
import os
import time
from dotenv import load_dotenv

# === CONFIG ===
BASE_DIR = r"D:\Android_Mobile_App\AndroidProject_5th"
OUTPUT_CSV = os.path.join(BASE_DIR, "github_android_search_results_2.csv")

# === Load GitHub token ===
load_dotenv("All_Tokens.env")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise ValueError("❌ GitHub token not found in .env")

headers = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}

# === Define star ranges ===
star_ranges = [
    (1, 100),
    (101, 300),
    (301, 600),
    (601, 1000),
    (1001, 5000),
    (5001, 10000),
    (10001, 20000)
]

# === Define queries for multiple languages ===
# Note: Each query string covers base keywords, language, and pushed date.
queries = [
    "android language:Kotlin pushed:>2024-01-01",
    "android language:Java pushed:>2024-01-01",
    "android language:Dart pushed:>2024-01-01"
]

# === Exclusion keywords ===
exclusion_keywords = ["sample", "tutorial"]

# === Search and collect ===
PER_PAGE = 100
PAGES = 10  # max 10 pages per star range

all_repos = []

for base_query in queries:
    for star_min, star_max in star_ranges:
        query = f"{base_query} stars:{star_min}..{star_max}"
        print(f"🔍 Query: {query}")

        for page in range(1, PAGES + 1):
            print(f"  📄 Fetching page {page} ...")
            url = f"https://api.github.com/search/repositories?q={query}&per_page={PER_PAGE}&page={page}"
            r = requests.get(url, headers=headers)

            if r.status_code == 403:
                print("⚠️ Rate limit hit. Sleeping 60 seconds...")
                time.sleep(60)
                continue

            r.raise_for_status()
            data = r.json()
            if "items" not in data or len(data["items"]) == 0:
                print(f"  ✅ No more results for this page.")
                break

            for item in data['items']:
                name = item['name'].lower()
                description = (item['description'] or "").lower()

                # Exclude if any keyword is found in name or description
                if any(kw in name or kw in description for kw in exclusion_keywords):
                    continue

                all_repos.append({
                    "full_name": item['full_name'],
                    "clone_url": item['clone_url'],
                    "stars": item['stargazers_count'],
                    "language": item['language'],
                    "pushed_at": item['pushed_at']
                })

            # Be nice to API
            time.sleep(1)

# === Deduplicate ===
df = pd.DataFrame(all_repos).drop_duplicates(subset="full_name").sort_values(by="stars", ascending=False)
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Finished! Total unique repos: {len(df)} saved to:\n{OUTPUT_CSV}")


🔍 Query: android language:Kotlin pushed:>2024-01-01 stars:1..100
  📄 Fetching page 1 ...
  📄 Fetching page 2 ...
  📄 Fetching page 3 ...
  📄 Fetching page 4 ...
  📄 Fetching page 5 ...
  📄 Fetching page 6 ...
  📄 Fetching page 7 ...
  📄 Fetching page 8 ...
  📄 Fetching page 9 ...
  📄 Fetching page 10 ...
🔍 Query: android language:Kotlin pushed:>2024-01-01 stars:101..300
  📄 Fetching page 1 ...
  📄 Fetching page 2 ...
  📄 Fetching page 3 ...
  📄 Fetching page 4 ...
  📄 Fetching page 5 ...
  📄 Fetching page 6 ...
  📄 Fetching page 7 ...
  📄 Fetching page 8 ...
  📄 Fetching page 9 ...
  📄 Fetching page 10 ...
  ✅ No more results for this page.
🔍 Query: android language:Kotlin pushed:>2024-01-01 stars:301..600
  📄 Fetching page 1 ...
  📄 Fetching page 2 ...
  📄 Fetching page 3 ...
  📄 Fetching page 4 ...
  📄 Fetching page 5 ...
  ✅ No more results for this page.
🔍 Query: android language:Kotlin pushed:>2024-01-01 stars:601..1000
  📄 Fetching page 1 ...
  📄 Fetching page 2 ...
  📄 Fetching 